In [1]:
import json
import math
import os
import functions_for_experiments
from openai import OpenAI
from sentence_transformers import SentenceTransformer, CrossEncoder
from qdrant_client import QdrantClient, models
from qdrant_client.models import SparseTextEmbedding
from thefuzz import fuzz
from fastembed import SparseTextEmbedding

c:\Users\Olena\Downloads\ucu-employee-support-rag\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 393/393 [00:00<00:00, 2129.06it/s]


In [2]:
with open("final_golden_dataset copy.json", "r", encoding="utf-8") as f:
    golden_data = json.load(f)

In [3]:
n = len(golden_data)

## Fixed chunking

In [4]:
MODEL_BASELINE = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
COLLECTION_BASELINE = "ucu_documents_baseline"

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2280.11it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
total_hit_baseline, total_mrr_baseline, total_recall_baseline, total_ndcg_baseline, \
total_hit_reranked_baseline, total_mrr_reranked_baseline, total_recall_reranked_baseline, total_ndcg_reranked_baseline = functions_for_experiments.get_metrics(MODEL_BASELINE, COLLECTION_BASELINE)

In [6]:
print("Results for baseline model:")
print(f"Hit Rate @ 5: {round(total_hit_baseline/n, 2)}")
print(f"MRR @ 5: {round(total_mrr_baseline/n, 2)}")
print(f"Recall @ 5: {round(total_recall_baseline/n, 2)}")
print(f"NDCG @ 5: {round(total_ndcg_baseline/n, 2)}")

Results for baseline model:
Hit Rate @ 5: 0.34
MRR @ 5: 0.21
Recall @ 5: 0.31
NDCG @ 5: 0.22


In [7]:
print("Results for baseline model (reranker added):")
print(f"Hit Rate @ 5: {round(total_hit_reranked_baseline/n, 2)}")
print(f"MRR @ 5: {round(total_mrr_reranked_baseline/n, 2)}")
print(f"Recall @ 5: {round(total_recall_reranked_baseline/n, 2)}")
print(f"NDCG @ 5: {round(total_ndcg_reranked_baseline/n, 2)}")

Results for baseline model (reranker added):
Hit Rate @ 5: 0.46
MRR @ 5: 0.42
Recall @ 5: 0.44
NDCG @ 5: 0.42


## Recursive chunking

In [5]:
COLLECTION_BASELINE_RECURSIVE_CHAR = "ucu_documents_baseline_recursive_char"

In [9]:
total_hit_baseline_2, total_mrr_baseline_2, total_recall_baseline_2, total_ndcg_baseline_2, \
total_hit_reranked_baseline_2, total_mrr_reranked_baseline_2, total_recall_reranked_baseline_2, total_ndcg_reranked_baseline_2 = functions_for_experiments.get_metrics(MODEL_BASELINE, COLLECTION_BASELINE_RECURSIVE_CHAR)

In [10]:
print("Results for baseline model (different chunking):")
print(f"Hit Rate @ 5: {round(total_hit_baseline_2/n, 2)}")
print(f"MRR @ 5: {round(total_mrr_baseline_2/n, 2)}")
print(f"Recall @ 5: {round(total_recall_baseline_2/n, 2)}")
print(f"NDCG @ 5: {round(total_ndcg_baseline_2/n, 2)}")

Results for baseline model (different chunking):
Hit Rate @ 5: 0.36
MRR @ 5: 0.28
Recall @ 5: 0.33
NDCG @ 5: 0.29


In [11]:
print("Results for baseline model (reranker added):")
print(f"Hit Rate @ 5: {round(total_hit_reranked_baseline_2/n, 2)}")
print(f"MRR @ 5: {round(total_mrr_reranked_baseline_2/n, 2)}")
print(f"Recall @ 5: {round(total_recall_reranked_baseline_2/n, 2)}")
print(f"NDCG @ 5: {round(total_ndcg_reranked_baseline_2/n, 2)}")

Results for baseline model (reranker added):
Hit Rate @ 5: 0.58
MRR @ 5: 0.54
Recall @ 5: 0.54
NDCG @ 5: 0.51


## HyDE

In [6]:
total_hit_baseline_hyde, total_mrr_baseline_hyde, total_recall_baseline_hyde, total_ndcg_baseline_hyde, \
total_hit_reranked_baseline_hyde, total_mrr_reranked_baseline_hyde, total_recall_reranked_baseline_hyde, total_ndcg_reranked_baseline_hyde = functions_for_experiments.get_metrics_hyde(MODEL_BASELINE, COLLECTION_BASELINE_RECURSIVE_CHAR)

In [7]:
print("Results for baseline model (HyDE):")
print(f"Hit Rate @ 5: {round(total_hit_baseline_hyde/n, 2)}")
print(f"MRR @ 5: {round(total_mrr_baseline_hyde/n, 2)}")
print(f"Recall @ 5: {round(total_recall_baseline_hyde/n, 2)}")
print(f"NDCG @ 5: {round(total_ndcg_baseline_hyde/n, 2)}")

Results for baseline model (HyDE):
Hit Rate @ 5: 0.36
MRR @ 5: 0.26
Recall @ 5: 0.32
NDCG @ 5: 0.27


In [8]:
print("Results for baseline model (HyDE):")
print(f"Hit Rate @ 5: {round(total_hit_reranked_baseline_hyde/n, 2)}")
print(f"MRR @ 5: {round(total_mrr_reranked_baseline_hyde/n, 2)}")
print(f"Recall @ 5: {round(total_recall_reranked_baseline_hyde/n, 2)}")
print(f"NDCG @ 5: {round(total_ndcg_reranked_baseline_hyde/n, 2)}")

Results for baseline model (HyDE):
Hit Rate @ 5: 0.42
MRR @ 5: 0.29
Recall @ 5: 0.35
NDCG @ 5: 0.27


## Query Transform

In [9]:
total_hit_baseline_transform, total_mrr_baseline_transform, total_recall_baseline_transform, total_ndcg_baseline_transform, \
total_hit_reranked_baseline_transform, total_mrr_reranked_baseline_transform, total_recall_reranked_baseline_transform, total_ndcg_reranked_baseline_transform = functions_for_experiments.get_metrics_query_transform(MODEL_BASELINE, COLLECTION_BASELINE_RECURSIVE_CHAR)

In [10]:
print("Results for baseline model (Query Transform):")
print(f"Hit Rate @ 5: {round(total_hit_baseline_transform/n, 2)}")
print(f"MRR @ 5: {round(total_mrr_baseline_transform/n, 2)}")
print(f"Recall @ 5: {round(total_recall_baseline_transform/n, 2)}")
print(f"NDCG @ 5: {round(total_ndcg_baseline_transform/n, 2)}")

Results for baseline model (Query Transform):
Hit Rate @ 5: 0.38
MRR @ 5: 0.27
Recall @ 5: 0.35
NDCG @ 5: 0.28


In [11]:
print("Results for baseline model (Query Transform):")
print(f"Hit Rate @ 5: {round(total_hit_reranked_baseline_transform/n, 2)}")
print(f"MRR @ 5: {round(total_mrr_reranked_baseline_transform/n, 2)}")
print(f"Recall @ 5: {round(total_recall_reranked_baseline_transform/n, 2)}")
print(f"NDCG @ 5: {round(total_ndcg_reranked_baseline_transform/n, 2)}")

Results for baseline model (Query Transform):
Hit Rate @ 5: 0.54
MRR @ 5: 0.51
Recall @ 5: 0.51
NDCG @ 5: 0.49


## Hybrid

In [15]:
MODEL_SPARSE = SparseTextEmbedding(model_name="Qdrant/bm25")

In [13]:
COLLECTION_BASELINE_HYBRID = "ucu_documents_baseline_hybrid"

In [20]:
total_hit_baseline_sparse, total_mrr_baseline_sparse, total_recall_baseline_sparse, total_ndcg_baseline_sparse, \
total_hit_reranked_baseline_sparse, total_mrr_reranked_baseline_sparse, total_recall_reranked_baseline_sparse, total_ndcg_reranked_baseline_sparse = functions_for_experiments.get_metrics_sparse(MODEL_BASELINE, COLLECTION_BASELINE_HYBRID, MODEL_SPARSE)

In [21]:
print("Results for baseline model (Hybrid):")
print(f"Hit Rate @ 5: {round(total_hit_baseline_sparse/n, 2)}")
print(f"MRR @ 5: {round(total_mrr_baseline_sparse/n, 2)}")
print(f"Recall @ 5: {round(total_recall_baseline_sparse/n, 2)}")
print(f"NDCG @ 5: {round(total_ndcg_baseline_sparse/n, 2)}")

Results for baseline model (Hybrid):
Hit Rate @ 5: 0.54
MRR @ 5: 0.43
Recall @ 5: 0.5
NDCG @ 5: 0.42


In [22]:
print("Results for baseline model (Hybrid):")
print(f"Hit Rate @ 5: {round(total_hit_reranked_baseline_sparse/n, 2)}")
print(f"MRR @ 5: {round(total_mrr_reranked_baseline_sparse/n, 2)}")
print(f"Recall @ 5: {round(total_recall_reranked_baseline_sparse/n, 2)}")
print(f"NDCG @ 5: {round(total_ndcg_reranked_baseline_sparse/n, 2)}")

Results for baseline model (Hybrid):
Hit Rate @ 5: 0.68
MRR @ 5: 0.64
Recall @ 5: 0.64
NDCG @ 5: 0.61


## Query Transform + Hybrid

In [16]:
total_hit_baseline_transform_hybrid, total_mrr_baseline_transform_hybrid, total_recall_baseline_transform_hybrid, total_ndcg_baseline_transform_hybrid, \
total_hit_reranked_baseline_transform_hybrid, total_mrr_reranked_baseline_transform_hybrid, total_recall_reranked_baseline_transform_hybrid, total_ndcg_reranked_baseline_transform_hybrid = functions_for_experiments.get_metrics_query_transform(MODEL_BASELINE, COLLECTION_BASELINE_HYBRID, sparse_model=MODEL_SPARSE)

In [17]:
print("Results for baseline model (Query Transform + Hybrid):")
print(f"Hit Rate @ 5: {round(total_hit_baseline_transform_hybrid/n, 2)}")
print(f"MRR @ 5: {round(total_mrr_baseline_transform_hybrid/n, 2)}")
print(f"Recall @ 5: {round(total_recall_baseline_transform_hybrid/n, 2)}")
print(f"NDCG @ 5: {round(total_ndcg_baseline_transform_hybrid/n, 2)}")

Results for baseline model (Query Transform + Hybrid):
Hit Rate @ 5: 0.44
MRR @ 5: 0.27
Recall @ 5: 0.4
NDCG @ 5: 0.29


In [18]:
print("Results for baseline model (Query Transform + Hybrid):")
print(f"Hit Rate @ 5: {round(total_hit_reranked_baseline_transform_hybrid/n, 2)}")
print(f"MRR @ 5: {round(total_mrr_reranked_baseline_transform_hybrid/n, 2)}")
print(f"Recall @ 5: {round(total_recall_reranked_baseline_transform_hybrid/n, 2)}")
print(f"NDCG @ 5: {round(total_ndcg_reranked_baseline_transform_hybrid/n, 2)}")

Results for baseline model (Query Transform + Hybrid):
Hit Rate @ 5: 0.62
MRR @ 5: 0.53
Recall @ 5: 0.58
NDCG @ 5: 0.52
